# 07 — Global admin1 & admin0 summary (all countries, per hazard)

Aggregates every country's admin2 hazard-exposure results up to **admin1** and **admin0**, per hazard, recomputing indices/classes at all valid normalisation scopes.

- **admin1** rows get **global + country + regional** scopes (`g/c/r` index, expi, expc). Country scope normalizes a country's admin1 units against each other.
- **admin0** rows get **global + regional** only — the unit *is* the country, so there is no country scope.

## Inputs

- Global per-hazard admin2 CSVs from notebook 02: `{SOURCE_FOLDER}/{hazard}_{date}.csv`
- Optional weight table from notebook 00: `{COUNTS_TABLE}`. Absent -> `hazs_mean/std` fall back to population weight.
- NA / state-country rules mirror notebook 02.

## Outputs

- `{OUTPUT_FOLDER}/global_admin1/CSV/{hazard}_{date}.csv`
- `{OUTPUT_FOLDER}/global_admin0/CSV/{hazard}_{date}.csv`

## Model

Counts summed; `hazs_min/max` via min/max; `hazs_mean/std` count-weighted pooled; `hazs_median` dropped. Log/relative inputs and every index/class recomputed from summed counts (`aggregation_functions` + `IndicatorUtils`). Only `status == ""` state-country units enter normalisation.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import os
import pandas as pd
import numpy as np

import aggregation_functions as agg
from indicator_functions import IndicatorUtils

iu = IndicatorUtils()
pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters
# ============================================================
PROD_DATE = "2026-08-31"
SOURCE_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali\global\CSV"
OUTPUT_FOLDER = "output"
COUNTS_TABLE = f"{OUTPUT_FOLDER}/aux/hazs_count_{PROD_DATE}.csv"  # optional
# Independent admin2 population source -- keeps the denominator identical for
# every hazard (see aggregation_functions.apply_population_reference).
POP_REFERENCE = f"{OUTPUT_FOLDER}/aux/admin2_population.csv"
PROCESS_SPECIFIC_HAZARDS = None  # e.g. ["river_flood_100yr"]

POP = agg.POPULATION_CLASSES

# State / NA rules — mirror notebook 02. Only state countries enter normalisation;
# non-state / NA / ID units are kept in the output with blank indices + a status.
state_countries = ['AFG_V1','AGO_V1','ALB_V1','AND_V1','ARE_V1','ARG_V1','ARM_V1','ATG_V2','AUS_V1','AUT_V1','AZE_V1','BDI_V1','BEL_V1','BEN_V1','BFA_V1','BGD_V1','BGR_V1','BHR_V2','BHS_V2','BIH_V1','BLR_V1','BLZ_V1','BOL_V1','BRA_V1','BRB_V2','BRN_V1','BTN_V1','BWA_V1','CAF_V1','CAN_V1','CHE_V1','CHL_V1','CHN_V1','CIV_V1','CMR_V1','COD_V1','COG_V1','COL_V1','COM_V2','CPV_V2','CRI_V1','CUB_V2','CYP_V1','CZE_V1','DEU_V1','DJI_V1','DMA_V2','DNK_V1','DOM_V2','DZA_V1','ECU_V1','EGY_V1','ERI_V1','ESP_V1','EST_V1','ETH_V1','FIN_V1','FJI_V2','FRA_V1','FSM_V2','GAB_V1','GBR_V1','GEO_V1','GHA_V1','GIN_V1','GMB_V1','GNB_V1','GNQ_V1','GRC_V1','GRD_V2','GTM_V1','GUY_V1','HND_V1','HRV_V1','HTI_V2','HUN_V1','IDN_V1','IND_V1','IRL_V1','IRN_V1','IRQ_V1','ISL_V1','ISR_V1','ITA_V1','JAM_V2','JOR_V1','JPN_V1','KAZ_V1','KEN_V1','KGZ_V1','KHM_V1','KIR_V2','KNA_V2','KOR_V1','KWT_V1','LAO_V1','LBN_V1','LBR_V1','LBY_V1','LCA_V2','LIE_V1','LKA_V1','LSO_V1','LTU_V1','LUX_V1','LVA_V1','MAR_V1','MCO_V1','MDA_V1','MDG_V1','MDV_V2','MEX_V1','MHL_V2','MKD_V1','MLI_V1','MLT_V1','MMR_V1','MNE_V1','MNG_V1','MOZ_V1','MRT_V1','MUS_V2','MWI_V1','MYS_V1','NAM_V1','NER_V1','NGA_V1','NIC_V1','NLD_V1','NOR_V1','NPL_V1','NRU_V2','NZL_V1','OMN_V1','PAK_V1','PAN_V1','PER_V1','PHL_V1','PLW_V2','PNG_V1','POL_V1','PRK_V1','PRT_V1','PRY_V1','PSE_V1','QAT_V1','ROU_V1','RUS_V1','RWA_V1','SAU_V1','SDN_V1','SEN_V1','SGP_V2','SLB_V2','SLE_V1','SLV_V1','SMR_V1','SOM_V1','SRB_V1','SSD_V1','STP_V2','SUR_V1','SVK_V1','SVN_V1','SWE_V1','SWZ_V1','SYC_V2','SYR_V1','TCD_V1','TGO_V1','THA_V1','TJK_V1','TKM_V1','TLS_V1','TON_V2','TTO_V2','TUN_V1','TUR_V1','TUV_V2','TZA_V1','UGA_V1','UKR_V1','URY_V1','USA_V1','UZB_V1','VAT_V1','VCT_V2','VEN_V1','VNM_V1','VUT_V2','WSM_V2','YEM_V1','ZAF_V1','ZMB_V1','ZWE_V1']
non_official_codes = ["ALA_V1","CCK_V1","CXR_V1","ESH_V1","GGY_V1","JEY_V1","NFK_V1","PCN_V1","SJM_V1"]
sovereignty_unsettled_countries = ['EGY1_V1','SDN1_V1','SSD1_V1','xAB_V1','xAC_V1','xAP_V1','xJK_V1','xJL_V1','xPI_V1','xRI_V1','xSI_V1','xSK_V1','xSR_V1','xxx_V1']
id_countries_insufficient_data = ["NIC_V1","PSE_V1","CUB_V2"]
id_countries_riverine = ["FJI_V2"]
id_countries_coastal_flood = ["CPV_V2","MDV_V2","MUS_V2","NRU_V2","WSM_V2","SYC_V2","TON_V2","TUV_V2"]
id_countries_fire = ["TUV_V2","MHL_V2","MDV_V2","NRU_V2","LIE_V1"]
id_countries_air_pollution = ["NRU_V2","MCO_V1"]

os.makedirs(f"{OUTPUT_FOLDER}/global_admin1/CSV", exist_ok=True)
os.makedirs(f"{OUTPUT_FOLDER}/global_admin0/CSV", exist_ok=True)
counts_df = agg.load_counts_table(COUNTS_TABLE)
pop_ref = pd.read_csv(POP_REFERENCE) if os.path.exists(POP_REFERENCE) else None
print("population reference:", f"loaded ({len(pop_ref):,} admin2)" if pop_ref is not None
      else "MISSING -> population read from hazard tables (drifts per hazard!)")
print("weight table:", "loaded" if counts_df is not None else "MISSING -> pop_total fallback")
codes = agg.discover_hazard_codes(SOURCE_FOLDER, PROD_DATE)
print(f"hazards: {len(codes)}")

In [ ]:
def apply_status_rules(df, hazard_code):
    """Assign the per-unit status flag at admin2 (before aggregation), mirroring
    notebook 02 but with the CORRECT status strings. A blank status ('') means
    the unit is usable and enters normalisation; any non-blank value keeps the
    unit in the output but excludes it from index/class computation.
    """
    df = df.copy()
    uc = df["adm0_ucode"]
    df["status"] = ""
    df.loc[uc.isin(non_official_codes + sovereignty_unsettled_countries), "status"] = "Not Applicable"
    df.loc[uc.isin(id_countries_insufficient_data), "status"] = "Insufficient Data"
    if hazard_code == "river_flood_100yr":
        df.loc[uc.isin(id_countries_riverine), "status"] = "Insufficient Data"
    if hazard_code == "coastal_flood_100yr":
        df.loc[uc.isin(id_countries_coastal_flood), "status"] = "Insufficient Data"
    if hazard_code in ("fire_intensity", "fire_frequency"):
        df.loc[uc.isin(id_countries_fire), "status"] = "Insufficient Data"
    if hazard_code == "air_pollution_pm25":
        df.loc[uc.isin(id_countries_air_pollution), "status"] = "Insufficient Data"
    # Non-state countries that fell through: keep but exclude from normalisation.
    df.loc[(df["status"] == "") & (~uc.isin(state_countries)), "status"] = "Not Applicable"
    # Remaining genuine No Data: both hazard and exposure null.
    nd = (df["status"] == "") & df["hazs_mean"].isnull() & df["aexp"].isnull()
    df.loc[nd, "status"] = "No Data"
    return df

In [ ]:
# Output column order (compact scheme, no hazs_median).
meta = ["iso3", "adm0_id", "adm0_name", "adm0_ucode"]
haz = ["hazs_mean", "hazs_std", "hazs_min", "hazs_max"]
rel = ["total_relative_exposure", "total_male_relative_exposure", "total_female_relative_exposure",
       "under_18_total_relative_exposure", "under_18_male_relative_exposure", "under_18_female_relative_exposure"]

def scope_cols(prefixes):
    cols = [f"{p}index" for p in prefixes]
    for p in prefixes:
        cols += [f"{p}expi_{pc}" for pc in POP]
    for p in prefixes:
        cols += [f"{p}expc_{pc}" for pc in POP]
    return cols

def assemble(df, level):
    key = ["adm1_name", "adm1_ucode", "unicef_region", "unicef_region_full"] if level == "adm1" else ["unicef_region", "unicef_region_full"]
    prefixes = ["g", "c", "r"] if level == "adm1" else ["g", "r"]
    # `n_children` from aggregate_counts counts the admin2 units pooled into
    # each parent -- NOT the under-18 population. Renamed to `n_admin2` to match
    # notebook 06 and config/column_labels.json ("# Admin2 Units"); the two
    # notebooks previously shipped the same quantity under different names.
    # n_admin2_covered sits directly after n_children (renamed n_admin2 below)
    # so the coverage count is always adjacent to its total.
    coverage = ["n_admin2_covered"] if "n_admin2_covered" in df.columns else []
    cols = meta + key + ["n_children"] + coverage + haz + agg.COUNT_COLS + rel + scope_cols(prefixes) + ["status"]
    out = df[[c for c in cols if c in df.columns]].copy()
    out = out.rename(columns={"n_children": "n_admin2"})
    return iu.round_cols(out)

In [ ]:
# ============================================================
# Per hazard: aggregate to admin1 (g/c/r) and admin0 (g/r)
# ============================================================
for code in codes:
    if PROCESS_SPECIFIC_HAZARDS and code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    print(f"\n=== {code} ===")
    df = agg.load_hazard_global(code, PROD_DATE, SOURCE_FOLDER)
    df = apply_status_rules(df, code)
    # Population from the independent reference BEFORE aggregating, so
    # suppressed units still contribute their people to the denominator.
    if pop_ref is not None:
        df = agg.apply_population_reference(df, pop_ref)
    df = agg.weight_table_join(df, counts_df, code)
    df = agg.recompute_exposure_inputs(df)

    # --- ADMIN1 ---
    a1 = agg.aggregate_counts(df, "adm1_ucode")
    a1 = agg.recompute_exposure_inputs(a1)
    a1 = agg.compute_scope(iu, a1, "g", None)             # global first (needed for borrow rule)
    a1 = agg.compute_scope(iu, a1, "c", "adm0_ucode")     # country
    a1 = agg.compute_scope(iu, a1, "r", "unicef_region")  # regional
    a1_out = assemble(a1, "adm1")
    a1_path = f"{OUTPUT_FOLDER}/global_admin1/CSV/{code}_{PROD_DATE}.csv"
    a1_out.to_csv(a1_path, index=False)

    # --- ADMIN0 ---
    a0 = agg.aggregate_counts(df, "adm0_ucode")
    a0 = agg.recompute_exposure_inputs(a0)
    a0 = agg.compute_scope(iu, a0, "g", None)
    a0 = agg.compute_scope(iu, a0, "r", "unicef_region")
    a0_out = assemble(a0, "adm0")
    a0_path = f"{OUTPUT_FOLDER}/global_admin0/CSV/{code}_{PROD_DATE}.csv"
    a0_out.to_csv(a0_path, index=False)

    print(f"  admin1: {len(a1_out)} units -> {a1_path}")
    print(f"  admin0: {len(a0_out)} units -> {a0_path}")

print("\nDone.")

In [ ]:
# ============================================================
# Verification (last processed hazard `code`)
# ============================================================
df = agg.load_hazard_global(code, PROD_DATE, SOURCE_FOLDER)
df = apply_status_rules(df, code)
usable = df[df["status"] == ""]
adm2_aexp = pd.to_numeric(usable["aexp"], errors="coerce").sum()
print(f"[{code}] conservation of aexp across levels:")
print(f"  admin2(usable)={adm2_aexp:,.0f}")
print(f"  admin1={a1_out['aexp'].sum():,.0f}  match={abs(adm2_aexp - a1_out['aexp'].sum()) < 1}")
print(f"  admin0={a0_out['aexp'].sum():,.0f}  match={abs(adm2_aexp - a0_out['aexp'].sum()) < 1}")
print("\nadmin0 rows == distinct state countries present:", a0_out.adm0_ucode.nunique())
for p in ["g", "c", "r"]:
    print(f"admin1 {p}expi_total in [0,10]:", a1_out[f"{p}expi_total"].dropna().between(0, 10).all(),
          f"| {p}expc_total classes:", sorted(a1_out[f"{p}expc_total"].dropna().unique()))
print("admin0 has no c-scope columns:", not any(cc.startswith("cexp") or cc == "cindex" for cc in a0_out.columns))